Instalação do Qdrant Client

In [1]:
%pip install qdrant-client fastembed

Conectando ao Qdrant Cloud

In [2]:
from qdrant_client import QdrantClient
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv()  # carrega o .env, para carregar a URL e a API key sem vazar elas

# connect to Qdrant Cloud
client = QdrantClient(
    url=os.getenv("QDRANT_URL"), # Adicionando o Endpoint Cluster, que foi obtido na etapa anterior
    api_key=os.getenv("QDRANT_API_KEY"), # Adicionando a API key, que foi obtida na etapa anterior
)

Criando a coleção

In [3]:
from qdrant_client.models import Distance, VectorParams

# create collection
client.create_collection( # envia uma requisição para o servidor criar uma nova coleção
    collection_name="items", # Define o nome da coleção
    vectors_config=VectorParams(size=384, distance=Distance.COSINE), # os vetores da coleção terão 384 dimensões e a similaridade entre eles é calculada com cosseno
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `items` already exists!"},"time":0.0275432}'

Aqui ocorreu esse erro 409, porque eu já tinha rodado todas as células antes, mas com a URL e a API key expostas, então coloquei o arquivo .env e executei todas novamente, portanto como já tinha criado a coleção "items" deu esse erro: "Collection `items` already exists!"

Populando a coleção

In [4]:
from qdrant_client.models import PointStruct
from fastembed import TextEmbedding

# load the embedding model
model = TextEmbedding('BAAI/bge-small-en-v1.5') # esse modelo 'BAAI/bge-small-en-v1.5' converte textos em vetores de 384 dimensões

menu_items = [ # Uma lista de items de um menu de algum restaurante, com nome, descrição, preço e categoria
    ("Pad Thai with Tofu", "Stir-fried rice noodles with tofu bean sprouts scallions and crushed peanuts in traditional tamarind sauce", "$13.95", "Noodles"),
    ("Grilled Salmon Fillet", "Wild-caught Atlantic salmon grilled with lemon butter and fresh herbs served with seasonal vegetables", "$24.50", "Seafood Entrees"),
    ("Mushroom Risotto", "Creamy arborio rice with mixed mushrooms parmesan truffle oil and fresh thyme", "$16.75", "Vegetarian"),
    ("Bibimbap Bowl", "Korean rice bowl with seasoned vegetables fried egg gochujang sauce and choice of protein", "$14.50", "Korean Bowls"),
    ("Falafel Wrap", "Crispy chickpea fritters with hummus tahini cucumber tomato and pickled vegetables in warm pita", "$11.25", "Mediterranean"),
    ("Shrimp Tacos", "Three soft tacos with grilled shrimp cabbage slaw chipotle aioli and fresh lime", "$13.00", "Tacos"),
    ("Vegetable Curry", "Mixed vegetables in aromatic coconut curry sauce with jasmine rice and naan bread", "$12.95", "Indian Curries"),
    ("Tuna Poke Bowl", "Fresh ahi tuna with avocado edamame cucumber seaweed salad over sushi rice with spicy mayo", "$16.50", "Poke Bowls"),
    ("Margherita Pizza", "Fresh mozzarella san marzano tomatoes basil and extra virgin olive oil on wood-fired crust", "$14.00", "Pizza"),
    ("Chicken Tikka Masala", "Tandoori chicken in creamy tomato sauce with aromatic spices served with basmati rice", "$15.95", "Indian Entrees"),
    ("Greek Salad", "Romaine lettuce tomatoes cucumbers kalamata olives feta cheese red onion with lemon oregano dressing", "$10.50", "Salads"),
    ("Lobster Roll", "Fresh Maine lobster meat with light mayo on toasted buttery roll served with chips", "$22.00", "Seafood Sandwiches"),
    ("Quinoa Buddha Bowl", "Organic quinoa with roasted chickpeas kale sweet potato tahini dressing and hemp seeds", "$13.50", "Healthy Bowls"),
    ("Beef Pho", "Traditional Vietnamese beef noodle soup with rice noodles fresh herbs bean sprouts and lime", "$12.75", "Noodle Soups"),
    ("Eggplant Parmesan", "Breaded eggplant layered with marinara mozzarella and parmesan served with pasta", "$15.25", "Italian Entrees"),
    ("Crab Cakes", "Maryland-style lump crab cakes with remoulade sauce and mixed greens", "$18.50", "Seafood Appetizers"),
    ("Tofu Stir Fry", "Crispy tofu with broccoli bell peppers snap peas in garlic ginger sauce over steamed rice", "$12.50", "Vegetarian Entrees"),
    ("Salmon Sushi Platter", "12 pieces of fresh salmon nigiri and sashimi with wasabi pickled ginger and soy sauce", "$19.95", "Sushi"),
    ("Caprese Sandwich", "Fresh mozzarella tomatoes basil pesto balsamic glaze on ciabatta bread", "$11.75", "Sandwiches"),
    ("Tom Yum Soup", "Spicy and sour Thai soup with shrimp lemongrass galangal mushrooms and kaffir lime leaves", "$11.50", "Soups"),
    ("Lentil Dal", "Red lentils simmered with turmeric cumin coriander served with rice and naan", "$11.95", "Vegan Entrees"),
    ("Fish and Chips", "Beer-battered cod with crispy fries malt vinegar and tartar sauce", "$16.00", "British Classics"),
    ("Veggie Burger", "House-made black bean and quinoa patty with avocado sprouts tomato on brioche bun", "$13.25", "Burgers"),
    ("Miso Ramen", "Rich miso broth with ramen noodles soft-boiled egg bamboo shoots nori and scallions", "$14.50", "Ramen"),
    ("Stuffed Bell Peppers", "Roasted bell peppers filled with rice vegetables herbs and melted cheese", "$13.75", "Vegetarian Entrees"),
    ("Scallop Risotto", "Pan-seared sea scallops over creamy parmesan risotto with white wine and lemon", "$26.50", "Seafood Specials"),
    ("Spring Rolls", "Fresh rice paper rolls with vegetables tofu rice noodles herbs and peanut dipping sauce", "$8.95", "Appetizers"),
    ("Oyster Po Boy", "Fried oysters with lettuce tomato pickles and remoulade on french bread", "$15.50", "Sandwiches"),
    ("Portobello Mushroom Steak", "Grilled portobello cap marinated in balsamic with roasted vegetables and quinoa", "$14.95", "Vegan Entrees"),
    ("Coconut Shrimp", "Jumbo shrimp breaded in shredded coconut served with sweet chili sauce", "$14.25", "Seafood Appetizers")
]

# embedding generator
points = []
embeddings = model.embed([f"{item[0]} {item[1]}" for item in menu_items]) # Gera embeddings para cada item do menu, concatena o nome (item[0]) com a descrição (item[1])
for i, embedding in enumerate(embeddings):
    vector = embedding.tolist() # Converte o embedding para uma lista Python, formato exigido pelo Qdrant
    point = PointStruct( # Cria um ponto que será armazenado no Qdrant
        id=i,
        vector=vector,
        payload={  # Define o payload, que são os metadados associados ao vetor
            "item_name": menu_items[i][0],
            "description": menu_items[i][1],
            "price": menu_items[i][2],
            "category": menu_items[i][3],
        }
    )
    points.append(point) # Adiciona o ponto criado à lista de pontos

# upsert points to collection
client.upsert( # Chama o método upsert do cliente Qdrant
  collection_name="items", # Especifica a coleção em que os pontos serão armazenados
  points=points, # Envia a lista de pontos
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

Pesquisando o menu de itens

In [5]:
# generate query embedding
query_text = "vegetarian dishes" # Texto da consulta que será usado para encontrar os itens semelhantes
query_vector = next(iter(model.embed(query_text))) # Gera o embedding do texto de consulta

# search for similar menu items
results = client.query_points( # Chama o método de busca por similaridade vetorial no Qdrant
    collection_name="items", # Define a coleção onde a busca será realizada
    query=query_vector, # Define o vector de consulta que será comparado com os vetores armazenados
    with_payload=True, # Define que os payloads dos metadados devem ser retornados
    limit=5
)

# print results
for result in results.points:
    print(f"Item: {result.payload.get('item_name', 'N/A')}")
    print(f"Score: {result.score}") # Imprime o score de similaridade entre o vetor de consulta e o vetor do item
    print(f"Description: {result.payload['description'][:150]}...") # Limita em 150 caracteres
    print(f"Price: {result.payload.get('price', 'N/A')}")
    print("---")

Item: Greek Salad
Score: 0.7652137
Description: Romaine lettuce tomatoes cucumbers kalamata olives feta cheese red onion with lemon oregano dressing...
Price: $10.50
---
Item: Falafel Wrap
Score: 0.747164
Description: Crispy chickpea fritters with hummus tahini cucumber tomato and pickled vegetables in warm pita...
Price: $11.25
---
Item: Tofu Stir Fry
Score: 0.7417176
Description: Crispy tofu with broccoli bell peppers snap peas in garlic ginger sauce over steamed rice...
Price: $12.50
---
Item: Vegetable Curry
Score: 0.73314714
Description: Mixed vegetables in aromatic coconut curry sauce with jasmine rice and naan bread...
Price: $12.95
---
Item: Salmon Sushi Platter
Score: 0.7281405
Description: 12 pieces of fresh salmon nigiri and sashimi with wasabi pickled ginger and soy sauce...
Price: $19.95
---
